In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install open_clip_torch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import open_clip
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


In [3]:
 
# ---------------- DEVICE ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# ---------------- CONFIG ----------------
BATCH_SIZE = 64
EPOCHS = 25
LR = 1e-5
NUM_CLASSES = 100
EPS = 1e-8
 
selected_classes = list(range(NUM_CLASSES))
 

In [4]:
# ---------------- CLIP ----------------
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_model = clip_model.to(device)
 
class CLIPImageEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model
 
    def forward(self, x):
        return self.clip.encode_image(x)
 
image_encoder = CLIPImageEncoder(clip_model).to(device)
 
if torch.cuda.device_count() > 1:
    image_encoder = nn.DataParallel(image_encoder)
 

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [5]:
# ---------------- TRANSFORMS ----------------
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    preprocess
])
 
transform_clean = preprocess
 

In [6]:
# ---------------- DATASET ----------------
def filter_dataset(dataset, selected_classes):
    targets = np.array(dataset.targets)
    mask = np.isin(targets, selected_classes)
 
    dataset.data = dataset.data[mask]
    dataset.targets = targets[mask].tolist()
 
    label_map = {cls: i for i, cls in enumerate(selected_classes)}
    dataset.targets = [label_map[t] for t in dataset.targets]
 
    return dataset
 
train_dataset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=transform_train
)
 
test_dataset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=transform_clean
)
 
train_eval_dataset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=False, transform=transform_clean
)
 
train_dataset = filter_dataset(train_dataset, selected_classes)
test_dataset = filter_dataset(test_dataset, selected_classes)
train_eval_dataset = filter_dataset(train_eval_dataset, selected_classes)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
 

100%|██████████| 169M/169M [00:04<00:00, 34.9MB/s] 


In [7]:
# ---------------- CLASSIFIER ----------------
feature_dim = clip_model.visual.output_dim
 
classifier = nn.Sequential(
    nn.Linear(feature_dim, 1024),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, NUM_CLASSES)
).to(device)
 
criterion = nn.CrossEntropyLoss()
 
optimizer = optim.AdamW([
    {"params": clip_model.parameters(), "lr": 1e-7},
    {"params": classifier.parameters(), "lr": 1e-3}
], weight_decay=1e-4)
 

In [8]:
# ---------------- NC + FLATNESS ----------------
def compute_nc_metrics(features, labels, W):
    features = features.cpu().numpy()
    labels = labels.cpu().numpy()
    W = W.detach().cpu().numpy()
 
    K = len(np.unique(labels))
 
    class_means = []
    for k in range(K):
        class_k = features[labels == k]
        class_means.append(np.mean(class_k, axis=0))
 
    class_means = np.array(class_means)
    overall_mean = np.mean(features, axis=0)
 
    # NC1
    Sw = 0
    for k in range(K):
        class_k = features[labels == k]
        Sw += np.sum((class_k - class_means[k]) ** 2)
 
    Sb = np.sum((class_means - overall_mean) ** 2)
    nc1 = Sw / (Sb + EPS)
 
    # NC2
    M = class_means - overall_mean
    M = M / (np.linalg.norm(M, axis=1, keepdims=True) + EPS)
    G = M @ M.T
 
    ETF = np.full((K, K), -1/(K-1))
    np.fill_diagonal(ETF, 1)
 
    nc2 = np.linalg.norm(G - ETF, ord='fro')
 
    # NC3
    Wn = W / (np.linalg.norm(W, axis=1, keepdims=True) + EPS)
    nc3 = np.linalg.norm(Wn - M, ord='fro')
 
    return nc1, nc2, nc3
 
 
def compute_flatness(features, logits, labels, W):
    probs = torch.softmax(logits, dim=1)
    p_y = probs[torch.arange(len(labels)), labels]
 
    curvature = p_y * (1 - p_y)
    feat_norm = torch.sum(features ** 2, dim=1)
 
    trace = torch.mean(feat_norm * curvature)
    W_norm = torch.sum(W ** 2)
 
    return (trace / (W_norm + EPS)).item()
 

In [9]:
# ---------------- TRACKERS ----------------
train_aug_hist = []
train_clean_hist = []
test_hist = []
 
nc1_hist, nc2_hist, nc3_hist = [], [], []
flatness_hist = []
 

In [ ]:
# ---------------- TRAIN LOOP ----------------
for epoch in range(EPOCHS):
 
    # ===== TRAIN (AUGMENTED) =====
    clip_model.train()
    classifier.train()
 
    train_preds, train_labels_all = [], []
 
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        images, labels = images.to(device), labels.to(device)
 
        optimizer.zero_grad()
 
        feats = image_encoder(images)
        feats = feats / feats.norm(dim=-1, keepdim=True)
 
        outputs = classifier(feats)
        loss = criterion(outputs, labels)
 
        loss.backward()
        optimizer.step()
 
        preds = outputs.argmax(dim=1)
 
        train_preds.extend(preds.cpu().numpy())
        train_labels_all.extend(labels.cpu().numpy())
 
    train_aug_acc = np.mean(np.array(train_preds) == np.array(train_labels_all)) * 100
    train_aug_hist.append(train_aug_acc)
 
    print(f"\nTrain (Aug): {train_aug_acc:.2f}%")
 
    # ===== TRAIN (CLEAN) =====
    clip_model.eval()
    classifier.eval()
 
    clean_preds, clean_labels = [], []
 
    with torch.no_grad():
        for images, labels in train_eval_loader:
            images, labels = images.to(device), labels.to(device)
 
            feats = image_encoder(images)
            feats = feats / feats.norm(dim=-1, keepdim=True)
 
            outputs = classifier(feats)
            preds = outputs.argmax(dim=1)
 
            clean_preds.extend(preds.cpu().numpy())
            clean_labels.extend(labels.cpu().numpy())
 
    train_clean_acc = np.mean(np.array(clean_preds) == np.array(clean_labels)) * 100
    train_clean_hist.append(train_clean_acc)
 
    print(f"Train (Clean): {train_clean_acc:.2f}%")
 
    # ===== TEST =====
    test_preds, test_labels_all = [], []
 
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
 
            feats = image_encoder(images)
            feats = feats / feats.norm(dim=-1, keepdim=True)
 
            outputs = classifier(feats)
            preds = outputs.argmax(dim=1)
 
            test_preds.extend(preds.cpu().numpy())
            test_labels_all.extend(labels.cpu().numpy())
 
    test_acc = np.mean(np.array(test_preds) == np.array(test_labels_all)) * 100
    test_hist.append(test_acc)
 
    print(f"Test: {test_acc:.2f}%")
 
    # ===== NC + FLATNESS (TRAIN CLEAN SET) =====
    all_feats, all_labels, all_logits = [], [], []
 
    with torch.no_grad():
        for images, labels in train_eval_loader:
            images, labels = images.to(device), labels.to(device)
 
            feats = image_encoder(images)
            feats = feats / feats.norm(dim=-1, keepdim=True)
 
            logits = classifier(feats)
 
            all_feats.append(feats)
            all_labels.append(labels)
            all_logits.append(logits)
 
    all_feats = torch.cat(all_feats)
    all_labels = torch.cat(all_labels)
    all_logits = torch.cat(all_logits)
 
    W = classifier[-1].weight
 
    nc1, nc2, nc3 = compute_nc_metrics(all_feats, all_labels, W)
    flatness = compute_flatness(all_feats, all_logits, all_labels, W)
 
    nc1_hist.append(nc1)
    nc2_hist.append(nc2)
    nc3_hist.append(nc3)
    flatness_hist.append(flatness)
 
    print(f"NC1: {nc1:.6f} | NC2: {nc2:.6f} | NC3: {nc3:.6f}")
    print(f"Flatness: {flatness:.6f}")

Epoch 1 [TRAIN]: 100%|██████████| 782/782 [06:03<00:00,  2.15it/s]


Train (Aug): 39.25%


Train (Clean): 68.96%
Test: 68.45%
NC1: 697.848267 | NC2: 24.589524 | NC3: 14.208362
Flatness: 0.000844


Epoch 2 [TRAIN]: 100%|██████████| 782/782 [06:09<00:00,  2.11it/s]


Train (Aug): 63.23%


...

Train (Clean): 89.72%
Test: 84.31%
NC1: 609.659119 | NC2: 20.851294 | NC3: 14.178308
Flatness: 0.000040


Epoch 24 [TRAIN]: 100%|██████████| 782/782 [06:10<00:00,  2.11it/s]


Train (Aug): 83.36%


Train (Clean): 89.75%
Test: 84.38%
NC1: 612.120667 | NC2: 20.760833 | NC3: 14.180839
Flatness: 0.000038


Epoch 25 [TRAIN]:  81%|████████  | 632/782 [04:59<01:10,  2.11it/s]